_If you have already completed this tutorial, you can run the following command to clean up your workspace, then move on to the next chapter._

In [ ]:
cd "$(dirname "$JPY_SESSION_NAME")"
rm -rf my_project pixi_demo_project my_pyworkspace

<img alt="pixi robot" src="images/pixi.png" width="250px" style="float: right"/>

## Here is `pixi`

`pixi` is a **modern, fast and reproducible package manager** built on the **conda package** collection.

### A bit of history

In the beginning, there was conda:

**In 2012**, Conda was released: **an open-source, cross-platform, multi-language package and environment manager** designed to make installing, managing and isolating software dependencies easier, especially for data scientists and researchers.

**Starting in 2015**, fully open-source, community-driven package channels were created:

- **conda-forge** for general-purpose packages
- **bioconda** for bioinformatics tools

_The adoption of conda by the Galaxy community greatly accelerated the packaging of bioinformatics tools._

**In 2019**, QuantStack, a French startup, introduced a much faster rewrite of conda: **mamba**.

Mamba notably offers **a better dependency solver**, **improved package download handling**, and a lighter **C++ implementation**.

### The limitations of conda/mamba

- **Centralized storage in the home directory**: packages and environments are stored centrally in the user’s home directory rather than alongside the projects that use them.
- **Environments disconnected from projects**: an environment lives independently of the project it serves, so nothing ties a given codebase to the exact environment it needs.
- **Sharing requires advanced configuration**: making an environment available to other users or across a team requires non-trivial setup.
- **Reproducibility is possible but complex**:
  - Environment export is imperfect: exported files often miss build details or contain platform-specific entries that prevent clean reinstallation.
  - Reliable reinstallation depends on third-party tools such as conda-lock.
- **Limited support for native Python packages**: installing packages from PyPI alongside conda packages is poorly integrated and can lead to conflicts.
- **Cross-platform support is mostly theoretical**: an environment defined on one operating system rarely installs as-is on another.
- **No built-in task mechanism**: unlike other package managers such as npm or cargo, conda provides no native way to define and run project tasks.

### The `pixi` revolution

**`pixi` appeared in 2023**, backed by a new startup: prefix.dev.

`pixi` is a new package management tool that fixes conda’s shortcomings while relying on the existing conda package repositories.

In other words, `pixi` offers a new approach to installing conda packages and managing software environments, while still using the existing conda packages.

**📁 Project-centric**
Designed to integrate fully with your projects.

**🔄 Reproducible**
Isolated, easily recreatable environments with built-in lock files.

**🌐 Cross-platform**
Works on Linux, macOS, Windows and more.

**🛠️ Tasks**
Manage complex pipelines effortlessly.

**🧩 Multi-environment**
Compose multiple environments in a single configuration file.

**🌍 Global tools**
Install global tools, safely isolated. Replaces apt, Homebrew, winget or condax.

**📦 Built on conda packages**
Uses standard conda packages and also supports Python packages.

## Getting pixi

`pixi` is already available on the Wave Research cluster so you don’t need to install it.

In [ ]:
module load pixi
module list

If you want to install `pixi` on your own computer, it comes as a single, statically linked binary executable with no external dependencies.
Pre-compiled binaries are available for all major operating systems: Linux, macOS and Windows.

```bash
# Installation on Linux and macOS
curl -fsSL https://pixi.sh/install.sh | sh
# Installation on Windows
powershell -ExecutionPolicy Bypass -c "irm -useb https://pixi.sh/install.ps1 | iex"
```

The installation script installs the pixi binary in `~/.pixi/bin`, then adds this folder to your PATH by editing your shell's startup script (for example `.bashrc`).

Pixi includes a built-in self-update command:

```bash
# Update pixi
pixi self-update
```

## Let's create our first pixi project

If you already have an existing project and you want to use pixi to manage its tools, you can just go into the project folder and run `pixi init`

For a new project, `pixi` can also create the project folder :

In [ ]:
pixi init my_project

pixi has initialised a new **pixi workspace** in the current folder. Basically, it means has created a default `pixi.toml` file. The folder also include default `.gitignore` and `.gitattributes` files to get it ready for version control with git.

### pixi.toml

This pixi.toml file is the minimal _manifest_ of your project. It contains global metadata about your project and two sections ready to describe the project dependencies and tasks.

In [61]:
cd my_project
cat pixi.toml | pygmentize -f terminal -l toml

bash: cd: my_project: No such file or directory
cat: pixi.toml: No such file or directory



### Installing software

To install a softwate in the workspace, you can use the `pixi add` command. Pixi will automatically understand that it has to be done in the current workspace folder.

In [ ]:
pixi add zenodo_get

`pixi` has installed the `zenodo_get` software in our workspace and it has also added it the workspace _manifest_ :

In [ ]:
cat pixi.toml | pygmentize -f terminal -l toml

### pixi.lock

A new `pixi.lock` has also been created in the workspace containing the detailed list of dependencies for the workspace. This file contains the detailed lists of packages that has been installed in the workspace.

It can be shared with along the `pixi.toml` file to let them installed the exact same tools set.

Each time a new tool is installed within the workspace, the `pixi.lock` will be updated.

In [ ]:
cat pixi.lock | head -20 | pygmentize -f terminal

### .pixi

All files are installed in a `.pixi` hidden folder that is fully local to the project.

### Running a tool

`zenodo_get` has been installed in a virtual environment inside the workspace.

To run it, you can use the `pixi run` command :

In [ ]:
pixi run zenodo_get --help

### pixi shell

If you don't want to use `pixi run` for each tools installed in your workspace, you can start a shell inside the workspace using `pixi shell`.

To stop the shell, just type `Ctlr`+`D` or `exit`

In [ ]:
pixi shell
zenodo_get --help
exit

### Adding a platform

By default, `pixi` installs the dependencies for the current OS. We can request to support more platform just by adding them to the project workspace.

In [ ]:
pixi workspace platform add osx-arm64

`pixi` will immediately resolve all dependencies for the new platform. It will trigger a notification if a dependency can't be resolved for one platform.

### Tasks

You can define common tasks in your project using `pixi task`.

For instance, let's add a task to download our project data from zenodo :

In [ ]:
pixi task add get_data "zenodo_get -o data -t 60 -R 3 123456/789101112"

The task has been added into our `pixi.toml` file.

In [ ]:
cat pixi.toml | pygmentize -f terminal -l toml

We can run it, using `pixi run` :

In [ ]:
pixi run get_data

_We set a fake DOI in the command line, so `zenodo_get` complains about the fact that the DOI can't be resolved._

## Retrive an existing pixi project

Let's try to download an existing project using pixi for dependencies management

First, download the sources of the project

In [ ]:
cd "$(dirname "$JPY_SESSION_NAME")"
git clone https://gitlab.com/ifb-elixirfr/training/tutorial-demo/pixi_demo_project.git
cd pixi_demo_project

Now we can list the tasks available

In [ ]:
pixi task list

When we run a task, `pixi` install all dependencies of the project automatically

In [ ]:
pixi run clean

## Using multiple environments with pixi

Pixi includes mechanisms for managing multiple environments within a single workspace.

### Key concepts

**Feature**: a feature defines part of an environment.
It can contain dependencies, tasks, channels, platforms, etc.
Multiple features can be defined within the same workspace.

**Environment**: an environment is a collection of features.
It can be activated to run tasks.
Multiple environments can be defined within the same workspace.

**Default**: every workspace contains a default environment. This environment contains the dependencies and tasks defined at the top level of the workspace.

Let's consider the following usecase: we want to test a python code on both python 3.11 and 3.12

![Python test environments](images/python-test-environments.png)

Create a new pixi workspace :

In [ ]:
cd "$(dirname "$JPY_SESSION_NAME")"
pixi init my_pyworkspace
cd my_pyworkspace

Create a new `test` feature that will need `pytest` as a requirement

In [ ]:
pixi add --feature test pytest

Declare a `test` task on the `test` feature that will run `pytest`

In [ ]:
pixi task add test --feature test pytest

Now create a new feature for each python version we want to support :

In [ ]:
pixi add --feature py311 python=3.11
pixi add --feature py312 python=3.12

Finally create an environment that will each rely on a python feature and the test feature :

In [ ]:
pixi workspace environment add test-py311 --feature py311 --feature test
pixi workspace environment add test-py312 --feature py312 --feature test

We can now run tests on python 3.11 or 3.12 just by choosing on wich environment we run the `test` task :

In [ ]:
pixi run --environment test-py311 test

In [ ]:
pixi run --environment test-py312 test

## In summary...

Key points:

- Pixi uses conda packages and channels, ensuring compatibility with the existing ecosystem.
- Pixi can also deploy Python packages, broadening its integration possibilities.
- The environment and workspace are installed directly in the project directory (`.pixi` folder).
- Pixi records declared dependencies in a manifest file and pins the exact installed versions in a lock file, making environments fully reproducible.
- Pixi lets you define project-wide tasks, simplifying workflow automation.

Pixi is the ideal solution for managing tool dependencies and bioinformatics tasks in a research project, combining simplicity and robustness.